In [18]:
# Cell 1 -- GPU Check
import torch

print(f'[INFO] PyTorch {torch.__version__}')
print(f'[INFO] CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'[INFO] GPU: {torch.cuda.get_device_name(0)}')
    print(f'[INFO] VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('[WARN] No GPU detected -- training will be slow on CPU')


[INFO] PyTorch 2.10.0+cpu
[INFO] CUDA available: False
[WARN] No GPU detected -- training will be slow on CPU


In [ ]:
# Cell 2 -- Configuration
from pathlib import Path
import torch, yaml

ROOT = Path('c:/Users/MSI/Music/Project_DigitalWitness')
VIDEO_ROOT = Path('c:/Users/MSI/Music/Dataset')
DATASET_DIR = ROOT / 'data' / 'dataset'
SEQ_DIR = ROOT / 'data' / 'lean_sequences'
MODELS_DIR = ROOT / 'models'
OUTPUTS_DIR = ROOT / 'outputs' / 'cases'

for d in [SEQ_DIR/'normal', SEQ_DIR/'shoplifting', MODELS_DIR, OUTPUTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

device      = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
NUM_WORKERS = 0 if device.type == 'cpu' else 2

# Read classes from data.yaml -- never hardcode
with open(DATASET_DIR / 'data.yaml') as f:
    _cfg = yaml.safe_load(f)
YOLO_CLASSES     = _cfg['names']
BEHAVIOR_CLASSES = ['normal', 'shoplifting']

# YOLO
YOLO_BASE = str(MODELS_DIR / 'yolo26n.pt')
YOLO_SAVE = str(MODELS_DIR / 'yolo_dw.pt')
EPOCHS_YOLO = 50
BATCH_YOLO = 16
FREEZE_YOLO = 10    # freeze backbone, train head only

# Sequence extraction
FPS_TARGET = 6      # 1 frame per ~0.17s -- catches sub-0.5s concealment
MAX_NORMAL_VIDEOS = 500    # cap to prevent UCF-Crime class imbalance

# BiLSTM
LEAN_FEAT_DIM = 16
LEAN_HIDDEN  = 128
LEAN_LAYERS = 2
LEAN_DROPOUT = 0.3
SEQ_LEN = 45        # 45 frames @ 6fps = 7.5s temporal window
SEQ_STRIDE = 15
EPOCHS_BILSTM = 20
LR_BILSTM = 5e-4
WEIGHT_DECAY = 1e-4
BATCH_BILSTM = 32
PATIENCE = 7
BILSTM_SAVE = str(MODELS_DIR / 'bilstm_lean_dw.pt')
BILSTM_INFO = str(MODELS_DIR / 'bilstm_lean_dw_info.json')

SMOKE_TEST = True   # True = 2-epoch check, 10 videos per class, runs in <5 min
if SMOKE_TEST:
    EPOCHS_YOLO, EPOCHS_BILSTM, MAX_NORMAL_VIDEOS = 2, 2, 10

print(f'Device: {device}  |  YOLO classes: {YOLO_CLASSES}')


Device: cpu  |  YOLO classes: ['looking-around', 'picking-holding', 'normal', 'shoplifting']


In [20]:
# Cell 3 -- YOLO Fine-tune
import shutil, yaml
from ultralytics import YOLO

# Patch data.yaml to absolute paths (mirrors reference notebook)
yaml_path = DATASET_DIR / 'data.yaml'
with open(yaml_path) as f:
    cfg = yaml.safe_load(f)
cfg.update({'train': str(DATASET_DIR/'train'/'images'),
            'val':   str(DATASET_DIR/'valid'/'images'),
            'test':  str(DATASET_DIR/'test'/'images')})
cfg.pop('path', None)
with open(yaml_path, 'w') as f:
    yaml.dump(cfg, f, sort_keys=False)

model = YOLO(YOLO_BASE)
results = model.train(
    data=str(yaml_path), epochs=EPOCHS_YOLO,
    imgsz=640, batch=BATCH_YOLO, freeze=FREEZE_YOLO,
    project=str(ROOT / 'runs'), name='yolo_dw',
    patience=10, save=True, plots=True,
    device=0 if device.type == 'cuda' else 'cpu'
)
src = Path(results.save_dir) / 'weights' / 'best.pt'
shutil.copy(src, YOLO_SAVE)
print(f'[INFO] YOLO saved -> {YOLO_SAVE}')


New https://pypi.org/project/ultralytics/8.4.33 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.11  Python-3.12.2 torch-2.10.0+cpu CPU (11th Gen Intel Core i5-1155G7 @ 2.50GHz)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=c:\Users\MSI\Music\Project_DigitalWitness\data\dataset\data.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=2, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=10, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=c:\Users\MSI\Music\Project_Digi

c:\Users\MSI\Music\Project_DigitalWitness\.venv\Lib\site-packages\polars\meta\build.py:5: UserWarning: Polars binary is missing!
  from polars._utils.polars_version import get_polars_version


Plotting labels to C:\Users\MSI\Music\Project_DigitalWitness\runs\yolo_dw\labels.jpg... 
WARNING name 'PySeries' is not defined
optimizer: 'optimizer=auto' found, ignoring 'lr0=0.01' and 'momentum=0.937' and determining best 'optimizer', 'lr0' and 'momentum' automatically... 
optimizer: AdamW(lr=0.00125, momentum=0.9) with parameter groups 114 weight(decay=0.0), 126 weight(decay=0.0005), 126 bias(decay=0.0)
Image sizes 640 train, 640 val
Using 0 dataloader workers
Logging results to C:\Users\MSI\Music\Project_DigitalWitness\runs\yolo_dw
Starting training for 2 epochs...

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
        1/2         0G      1.584      4.914    0.01919         69        640: 43% ━━━━━─────── 326/751 8.1s/it 28:26<57:3626


KeyboardInterrupt: 

In [ ]:
# Cell 4 -- YOLO Evaluation
import supervision as sv
from supervision.metrics import MeanAveragePrecision
from ultralytics import YOLO

model = YOLO(YOLO_SAVE)
ds = sv.DetectionDataset.from_yolo(
    images_directory_path=str(DATASET_DIR / 'test' / 'images'),
    annotations_directory_path=str(DATASET_DIR / 'test' / 'labels'),
    data_yaml_path=str(DATASET_DIR / 'data.yaml')
)
preds, targets = [], []
for _, image, target in ds:
    preds.append(sv.Detections.from_ultralytics(model(image, verbose=False)[0]))
    targets.append(target)

mAP = MeanAveragePrecision().update(preds, targets).compute()
print(f'mAP50: {mAP.map50:.3f}  |  mAP50-95: {mAP.map50_95:.3f}')
mAP.plot()


In [ ]:
# Cell 5 -- Extract Feature Sequences
import cv2, numpy as np, random
from pathlib import Path
from ultralytics import YOLO
from tqdm import tqdm


def extract_lean_features(results, img_w: int, img_h: int) -> np.ndarray:
    """YOLO result for one frame -> 16-dim float32 feature vector."""
    feat = np.zeros(16, dtype=np.float32)
    feat[8] = feat[9] = 0.5   # default centre

    if results.boxes is None or len(results.boxes) == 0:
        return feat

    cls_ids = results.boxes.cls.cpu().numpy().astype(int)
    confs = results.boxes.conf.cpu().numpy()
    xyxy = results.boxes.xyxy.cpu().numpy()

    # Map class names -> indices at runtime from YOLO_CLASSES
    name_to_idx = {n: i for i, n in enumerate(YOLO_CLASSES)}
    i_shop = name_to_idx.get('shoplifting',    3)
    i_look = name_to_idx.get('looking-around', 1)
    i_pick = name_to_idx.get('picking-holding',2)
    i_norm = name_to_idx.get('normal',         0)

    max_conf = {c: 0.0 for c in range(len(YOLO_CLASSES))}
    cnt = {c: 0   for c in range(len(YOLO_CLASSES))}
    for c, cf in zip(cls_ids, confs):
        max_conf[c] = max(max_conf[c], float(cf))
        cnt[c] += 1

    feat[0] = max_conf[i_shop]
    feat[1] = max_conf[i_look]
    feat[2] = max_conf[i_pick]
    feat[3] = max_conf[i_norm]
    feat[4] = min(cnt[i_shop] / 5, 1.0)
    feat[5] = min(cnt[i_look] / 5, 1.0)
    feat[6] = min(cnt[i_pick] / 5, 1.0)
    feat[7] = min(len(cls_ids) / 10, 1.0)

    best = int(np.argmax(confs))
    x1,y1,x2,y2 = xyxy[best]
    feat[8] = ((x1+x2)/2) / img_w
    feat[9] = ((y1+y2)/2) / img_h
    feat[10] = (x2-x1) / img_w
    feat[11] = (y2-y1) / img_h
    feat[12] = 1.0
    feat[13] = 1.0 if max_conf[i_shop] > 0 or max_conf[i_look] > 0 else 0.0
    feat[14] = 1.0 if max_conf[i_shop] > 0.5 else 0.0
    feat[15] = (cnt[i_shop]+cnt[i_look]) / max(len(cls_ids), 1)
    return feat


def process_video(yolo, video_path: Path, out_path: Path) -> bool:
    if out_path.exists(): return True
    cap = cv2.VideoCapture(str(video_path))
    src_fps = cap.get(cv2.CAP_PROP_FPS) or 25
    step = max(1, int(src_fps / FPS_TARGET))
    w, h = int(cap.get(3)), int(cap.get(4))
    feats, fi = [], 0
    while True:
        ret, frame = cap.read()
        if not ret: break
        if fi % step == 0:
            feats.append(extract_lean_features(yolo.predict(frame, verbose=False)[0], w, h))
        fi += 1
    cap.release()
    if len(feats) < SEQ_LEN: return False
    np.save(out_path, np.array(feats, dtype=np.float32))
    return True


yolo = YOLO(YOLO_SAVE); yolo.fuse()

for cls in BEHAVIOR_CLASSES:
    videos = list((VIDEO_ROOT / cls).glob('*.mp4')) + list((VIDEO_ROOT / cls).glob('*.avi'))
    if cls == 'normal' and MAX_NORMAL_VIDEOS:
        random.seed(42); videos = random.sample(videos, min(MAX_NORMAL_VIDEOS, len(videos)))
    ok = sum(process_video(yolo, v, SEQ_DIR/cls/(v.stem+'.npy'))
             for v in tqdm(videos, desc=cls))
    print(f'[INFO] {cls}: {ok}/{len(videos)} sequences saved')


In [ ]:
# Cell 6 -- Train LeanBiLSTM
import torch.nn as nn, torch.optim as optim, json
from torch.utils.data import Dataset, DataLoader, Subset, WeightedRandomSampler
from sklearn.model_selection import train_test_split


class TemporalAttention(nn.Module):
    def __init__(self, dim: int):
        super().__init__()
        self.attn = nn.Linear(dim, 1)
    def forward(self, x):
        w = torch.softmax(self.attn(x), dim=1)
        return (w * x).sum(1), w.squeeze(-1)


class LeanBiLSTM(nn.Module):
    def __init__(self, feat_dim=16, hidden=128, layers=2, dropout=0.3, n_cls=2):
        super().__init__()
        self.bilstm = nn.LSTM(feat_dim, hidden, layers, batch_first=True,
                                 bidirectional=True, dropout=dropout if layers>1 else 0)
        self.attention = TemporalAttention(hidden * 2)
        self.clf  = nn.Sequential(
            nn.LayerNorm(hidden*2), nn.Dropout(dropout),
            nn.Linear(hidden*2, 64), nn.ReLU(), nn.Linear(64, n_cls)
        )
    def forward(self, x):
        out, _ = self.bilstm(x)
        ctx, w = self.attention(out)
        return self.clf(ctx), w


class LeanSeqDataset(Dataset):
    def __init__(self, seq_dir, classes, seq_len, stride):
        self.items = []
        for label, cls in enumerate(classes):
            for fp in Path(seq_dir, cls).glob('*.npy'):
                arr = np.load(fp)
                for s in range(0, len(arr)-seq_len+1, stride):
                    self.items.append((arr[s:s+seq_len], label))
    def __len__(self): return len(self.items)
    def __getitem__(self, i):
        a, l = self.items[i]; return torch.FloatTensor(a), l


ds     = LeanSeqDataset(SEQ_DIR, BEHAVIOR_CLASSES, SEQ_LEN, SEQ_STRIDE)
labels = [it[1] for it in ds.items]
tr_i, va_i = train_test_split(range(len(ds)), test_size=0.2, stratify=labels, random_state=42)

counts = [labels.count(0), labels.count(1)]
w_samp = WeightedRandomSampler([1/counts[labels[i]] for i in tr_i], len(tr_i), True)
tr_dl = DataLoader(Subset(ds, tr_i), BATCH_BILSTM, sampler=w_samp, num_workers=NUM_WORKERS)
va_dl = DataLoader(Subset(ds, va_i), BATCH_BILSTM, shuffle=False,  num_workers=NUM_WORKERS)

model = LeanBiLSTM(LEAN_FEAT_DIM, LEAN_HIDDEN, LEAN_LAYERS, LEAN_DROPOUT).to(device)
cw = torch.FloatTensor([max(counts)/c for c in counts]).to(device)
crit = nn.CrossEntropyLoss(weight=cw)
opt = optim.AdamW(model.parameters(), lr=LR_BILSTM, weight_decay=WEIGHT_DECAY)
sched = optim.lr_scheduler.StepLR(opt, step_size=8, gamma=0.5)

best, no_imp = 0.0, 0
for ep in range(1, EPOCHS_BILSTM+1):
    model.train(); tl=tc=tt=0
    for x,y in tr_dl:
        x,y = x.to(device),y.to(device); opt.zero_grad()
        lg,_ = model(x); loss = crit(lg,y); loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step()
        tl+=loss.item()*len(x); tc+=(lg.argmax(1)==y).sum().item(); tt+=len(x)

    model.eval(); vl=vc=vt=0
    with torch.no_grad():
        for x,y in va_dl:
            x,y=x.to(device),y.to(device); lg,_=model(x)
            vl+=crit(lg,y).item()*len(x); vc+=(lg.argmax(1)==y).sum().item(); vt+=len(x)

    ta,va = tc/tt, vc/vt
    print(f'Ep {ep:3d} | train={ta:.3f} val={va:.3f} lr={sched.get_last_lr()[0]:.1e}')
    sched.step()
    if va > best:
        best=va; torch.save(model.state_dict(), BILSTM_SAVE); no_imp=0; print('  saved')
    else:
        no_imp+=1
        if no_imp>=PATIENCE: print(f'Early stop ep {ep}'); break

json.dump({'pipeline':'lean_b','feat_dim':LEAN_FEAT_DIM,'hidden':LEAN_HIDDEN,
           'seq_len':SEQ_LEN,'best_val_acc':best,'classes':BEHAVIOR_CLASSES},
          open(BILSTM_INFO,'w'), indent=2)
print(f'[INFO] Best val accuracy: {best:.3f}')


In [ ]:
# Cell 7 -- Evaluate BiLSTM
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

model.load_state_dict(torch.load(BILSTM_SAVE, map_location=device))
model.eval()
all_p, all_y = [], []
with torch.no_grad():
    for x, y in va_dl:
        lg, _ = model(x.to(device))
        all_p.extend(lg.argmax(1).cpu().tolist())
        all_y.extend(y.tolist())

print(classification_report(all_y, all_p, target_names=BEHAVIOR_CLASSES))

cm = confusion_matrix(all_y, all_p)
fig, ax = plt.subplots(figsize=(5,4))
ConfusionMatrixDisplay(cm, display_labels=BEHAVIOR_CLASSES).plot(ax=ax, cmap='Blues')
plt.tight_layout()
plt.savefig(str(OUTPUTS_DIR / 'lean_confusion.png'), dpi=150)
plt.show()


In [ ]:
# Cell 8 -- End-to-End Smoke Test
# Test full inference on one video file
TEST_VIDEO = str(VIDEO_ROOT / 'shoplifting' / list((VIDEO_ROOT/'shoplifting').glob('*.mp4'))[0].name)

yolo = YOLO(YOLO_SAVE); yolo.fuse()
model = LeanBiLSTM(LEAN_FEAT_DIM, LEAN_HIDDEN, LEAN_LAYERS, LEAN_DROPOUT).to(device)
model.load_state_dict(torch.load(BILSTM_SAVE, map_location=device)); model.eval()

cap = cv2.VideoCapture(TEST_VIDEO)
fps_src = cap.get(cv2.CAP_PROP_FPS) or 25
step = max(1, int(fps_src / FPS_TARGET))
w, h     = int(cap.get(3)), int(cap.get(4))
feats, fi = [], 0
while True:
    ret, frame = cap.read()
    if not ret: break
    if fi % step == 0:
        feats.append(extract_lean_features(yolo.predict(frame, verbose=False)[0], w, h))
    fi += 1
cap.release()

if len(feats) >= SEQ_LEN:
    seq   = torch.FloatTensor(feats[:SEQ_LEN]).unsqueeze(0).to(device)
    with torch.no_grad(): logits, attn = model(seq)
    probs = torch.softmax(logits, dim=1)[0]
    pred  = BEHAVIOR_CLASSES[logits.argmax(1).item()]
    print(f'[RESULT] {pred.upper()} -- normal:{probs[0]:.2%}  shoplifting:{probs[1]:.2%}')
    print(f'[ATTN]   Top 3 frames: {attn[0].topk(3).indices.tolist()}')
else:
    print(f'[WARN] Video too short: only {len(feats)} frames extracted')
